# Read XML from AWS S3

This notebook reads an XML file from an S3 bucket using PySpark's built-in XML data source (`spark-xml`), which is pre-installed in the Databricks runtime.

In [0]:
# S3 path to the directory containing all XML files
s3_path = "s3://bundesliga-2022-2023-data/matchinformation/"

# The XML structure is:
# <PutDataRequest ...>
#   <MatchInformation>
#     <General ... />          <- match metadata (competition, teams, result, etc.)
#     <Environment ... />      <- stadium & weather info
#     <Teams>
#       <Team ...>             <- each team
#         <Players><Player ... /></Players>
#         <TrainerStaff>...</TrainerStaff>
#       </Team>
#     </Teams>
#   </MatchInformation>
# </PutDataRequest>

# Read ALL XML files in the directory at the MatchInformation level
# Spark will read every .xml file in the folder as a separate match
match_info_df = (
    spark.read.format("xml")
    .option("rowTag", "MatchInformation")
    .option("encoding", "UTF-8")
    .load(s3_path)
)

print(f"Total matches loaded: {match_info_df.count()}")
match_info_df.printSchema()

In [0]:
# Display a sample of the parsed XML data
display(match_info_df.limit(20))

In [0]:
from pyspark.sql.functions import col, explode

# ── 1. Flatten match-level info (1 row) ──
match_flat_df = match_info_df.select(
    # General
    col("General._MatchId").alias("match_id"),
    col("General._CompetitionName").alias("competition_name"),
    col("General._CompetitionId").alias("competition_id"),
    col("General._MatchDay").alias("match_day"),
    col("General._Season").alias("season"),
    col("General._SeasonId").alias("season_id"),
    col("General._HomeTeamName").alias("home_team"),
    col("General._HomeTeamId").alias("home_team_id"),
    col("General._GuestTeamName").alias("guest_team"),
    col("General._GuestTeamId").alias("guest_team_id"),
    col("General._Result").cast("string").alias("result"),
    col("General._MatchTitle").alias("match_title"),
    col("General._KickoffTime").alias("kickoff_time"),
    col("General._PlannedKickoffTime").alias("planned_kickoff_time"),
    col("General._TypeOfSport").alias("type_of_sport"),
    col("General._Type").alias("match_type"),
    col("General._Host").alias("host"),
    # Environment
    col("Environment._StadiumName").alias("stadium"),
    col("Environment._StadiumId").alias("stadium_id"),
    col("Environment._Country").alias("country"),
    col("Environment._NumberOfSpectators").alias("spectators"),
    col("Environment._StadiumCapacity").alias("stadium_capacity"),
    col("Environment._SoldOut").alias("sold_out"),
    col("Environment._Temperature").alias("temperature"),
    col("Environment._AirHumidity").alias("air_humidity"),
    col("Environment._AirPressure").alias("air_pressure"),
    col("Environment._Precipitation").alias("precipitation"),
    col("Environment._Roof").alias("roof"),
    col("Environment._Floodlight").alias("floodlight"),
    col("Environment._NeutralVenue").alias("neutral_venue"),
    col("Environment._PitchX").alias("pitch_x"),
    col("Environment._PitchY").alias("pitch_y"),
    # OtherGameInformation
    col("OtherGameInformation._PlayingTimeFirstHalf").alias("playing_time_first_half"),
    col("OtherGameInformation._PlayingTimeSecondHalf").alias("playing_time_second_half"),
    col("OtherGameInformation._TotalTimeFirstHalf").alias("total_time_first_half"),
    col("OtherGameInformation._TotalTimeSecondHalf").alias("total_time_second_half"),
)

display(match_flat_df)

In [0]:
# ── 2. Flatten players (one row per player, with match + team context) ──
players_df = (
    match_info_df
    .select(
        col("General._MatchId").alias("match_id"),
        col("General._CompetitionName").alias("competition_name"),
        col("General._MatchDay").alias("match_day"),
        col("General._Season").alias("season"),
        col("General._HomeTeamName").alias("home_team"),
        col("General._GuestTeamName").alias("guest_team"),
        col("General._Result").cast("string").alias("result"),
        col("General._KickoffTime").alias("kickoff_time"),
        explode("Teams.Team").alias("team"),
    )
    .select(
        "match_id", "competition_name", "match_day", "season",
        "home_team", "guest_team", "result", "kickoff_time",
        col("team._TeamId").alias("team_id"),
        col("team._TeamName").alias("team_name"),
        col("team._Role").alias("team_role"),
        col("team._LineUp").alias("line_up"),
        explode("team.Players.Player").alias("player"),
    )
    .select(
        "match_id", "competition_name", "match_day", "season",
        "home_team", "guest_team", "result", "kickoff_time",
        "team_id", "team_name", "team_role", "line_up",
        col("player._PersonId").alias("person_id"),
        col("player._FirstName").alias("first_name"),
        col("player._LastName").alias("last_name"),
        col("player._Shortname").alias("short_name"),
        col("player._ShirtNumber").alias("shirt_number"),
        col("player._PlayingPosition").alias("playing_position"),
        col("player._Starting").alias("starting"),
        col("player._TeamLeader").alias("team_leader"),
    )
)

display(players_df)

In [0]:
# ── 3. Flatten referees (one row per referee) ──
referees_df = (
    match_info_df
    .select(
        col("General._MatchId").alias("match_id"),
        col("General._CompetitionName").alias("competition_name"),
        col("General._MatchDay").alias("match_day"),
        col("General._Season").alias("season"),
        explode("Referees.Referee").alias("referee"),
    )
    .select(
        "match_id", "competition_name", "match_day", "season",
        col("referee._PersonId").alias("person_id"),
        col("referee._FirstName").alias("first_name"),
        col("referee._LastName").alias("last_name"),
        col("referee._Shortname").alias("short_name"),
        col("referee._Role").alias("role"),
    )
)

display(referees_df)

In [0]:
# ── 4. Flatten trainer staff (one row per staff member, with team context) ──
staff_df = (
    match_info_df
    .select(
        col("General._MatchId").alias("match_id"),
        col("General._CompetitionName").alias("competition_name"),
        col("General._MatchDay").alias("match_day"),
        col("General._Season").alias("season"),
        explode("Teams.Team").alias("team"),
    )
    .select(
        "match_id", "competition_name", "match_day", "season",
        col("team._TeamId").alias("team_id"),
        col("team._TeamName").alias("team_name"),
        col("team._Role").alias("team_role"),
        explode("team.TrainerStaff.Trainer").alias("staff"),
    )
    .select(
        "match_id", "competition_name", "match_day", "season",
        "team_id", "team_name", "team_role",
        col("staff._PersonId").alias("person_id"),
        col("staff._FirstName").alias("first_name"),
        col("staff._LastName").alias("last_name"),
        col("staff._Shortname").alias("short_name"),
        col("staff._Role").alias("role"),
    )
)

display(staff_df)

In [0]:
# Save all flattened DataFrames as Delta tables
# Using "overwrite" since all 7 files (including the one already loaded) are now read together.
# For future incremental file additions, switch mode to "append".

catalog = "`bundesliga-2022-2023`"
schema_name = f"{catalog}.batch"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

# 1. Match-level info
match_flat_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.match_info")

# 2. Players (one row per player)
players_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.players")

# 3. Referees (one row per referee)
referees_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.referees")

# 4. Trainer staff (one row per staff member)
staff_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.staff")

print("Tables created:")
for t in ["match_info", "players", "referees", "staff"]:
    count = spark.table(f"{schema_name}.{t}").count()
    print(f"  {schema_name}.{t}: {count} rows")